# Notebook 04 - Train the Consensus MLP (23-dim numerical features)

This notebook trains the **consensus fusion head**. After Agent A (EfficientNet-B4) and Agent B (ViT-B/16) classify a lesion and the debate trigger fires, the consensus MLP fuses the two agents' signals into a single calibrated 8-class prediction.

The previous 788-dim Groq-LLM-debate-text + sentence-embedding contract has been **removed entirely** and replaced with a fixed **23-dimensional pure numerical** feature vector (identical to `backend/ml/debate/features.py` and `05_evaluation.ipynb`):

| Component | Dim | Meaning |
|-----------|-----|---------|
| `pA` | 8 | Agent A softmax probabilities |
| `pB` | 8 | Agent B softmax probabilities |
| disagreement | 4 | JS divergence, entropy_A, entropy_B, max prob delta |
| attention | 3 | attn IoU, attn entropy_A, attn entropy_B (0.0 if unavailable) |

Reasons for the change: Groq rate-limited after 2-3 images (unusable for thousands of calls); ~100% of previous arguments were deterministic fallback templates, not real LLM output; 788 features with ~1,500 samples is a statistically indefensible 1.9:1 ratio, whereas 23 features give ~65:1. The architecture and parameter names match `backend/ml/consensus/classifier.py` so the checkpoint loads in the app.

## Exact Kaggle setup (do this before Run All)
1. **Attach the data**: Add Data -> search **`andrewmvd/isic-2019`** -> Add.
2. **Attach the previous checkpoints**: the outputs of Notebook 01 (`agent_a_best.pth`) and Notebook 02 (`agent_b_best.pth`). `find_file()` locates them under `/kaggle/input`.
3. **Accelerator**: Settings -> Accelerator -> **GPU T4 x2** (or **P100**).
4. **Internet**: Settings -> Internet -> **ON** (pip + pretrained timm weights). No Groq / sentence-transformers / API key is needed any more.
5. **Run All**.

> Outputs: `consensus_best.pth`, `consensus_scaler.pkl` (+ `.json` sidecar) and `consensus_temperature.txt`.

## 1. Install extras, imports, shared helpers and device

In [ ]:
# Kaggle pre-installs torch, torchvision, numpy, pandas, scikit-learn, matplotlib,
# seaborn, opencv-python, Pillow, tqdm. We install the EXTRAS this notebook needs.
# INTERNET MUST BE ON (Settings -> Internet -> ON) for pip + pretrained weights.
import sys, subprocess

print("Installing extras (timm, grad-cam, netcal)...")
print("NOTE: Internet must be ON for this to work (Settings -> Internet -> ON). No Groq / sentence-transformers / API key is used.")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-U", "timm",                 # -U so the ViT model id resolves
    "grad-cam",                   # Grad-CAM++ for Agent A
    "netcal",                     # ECE metric
], check=False)

import os, json, math, time, warnings, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
import joblib
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# --- Shared contract constants (identical to backend + other notebooks) ----
ISIC_CLASSES = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMAGE_SIZE = 224
NUM_CLASSES = 8

# Consensus feature contract (must match backend/ml/debate/features.py and 05_evaluation.ipynb).
FEATURE_DIM = 23

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Reproducibility.
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


def discover_isic(root="/kaggle/input"):
    """Locate the ISIC-2019 ground-truth CSV and the training-image directory.

    The CSV is the .csv whose header contains ALL 8 ISIC class names. The image
    dir is the directory containing the most ISIC_*.jpg/.jpeg files. Robust to
    nested mirror folders (e.g. doubled ISIC_2019_Training_Input/).
    """
    csv_path = None
    class_set = set(ISIC_CLASSES)
    for dirpath, _dirnames, filenames in os.walk(root):
        for fn in filenames:
            if fn.lower().endswith(".csv"):
                fp = os.path.join(dirpath, fn)
                try:
                    cols = set(pd.read_csv(fp, nrows=0).columns)
                except Exception:
                    continue
                if class_set.issubset(cols):
                    csv_path = fp
                    break
        if csv_path is not None:
            break

    # Image dir = directory with the most ISIC_*.jpg/.jpeg files.
    best_dir = None
    best_count = 0
    for dirpath, _dirnames, filenames in os.walk(root):
        count = 0
        for fn in filenames:
            low = fn.lower()
            if low.startswith("isic_") and (low.endswith(".jpg") or low.endswith(".jpeg")):
                count += 1
        if count > best_count:
            best_count = count
            best_dir = dirpath

    print("Discovered ground-truth CSV :", csv_path)
    print("Discovered image directory  :", best_dir, "(%d ISIC images)" % best_count)
    assert csv_path is not None and os.path.exists(csv_path), "Could not find ISIC ground-truth CSV under " + root
    assert best_dir is not None and best_count > 0, "Could not find an ISIC image directory under " + root
    return csv_path, best_dir


def find_file(filename_substring, search_roots=("/kaggle/input", "/kaggle/working")):
    """Return the first path whose basename contains the given substring."""
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _dirnames, filenames in os.walk(root):
            for fn in filenames:
                if filename_substring in fn:
                    found = os.path.join(dirpath, fn)
                    print("find_file('%s') -> %s" % (filename_substring, found))
                    return found
    print("find_file('%s'): NOT FOUND under %s" % (filename_substring, list(search_roots)))
    return None


## 3. Load Agent A and Agent B

Rebuild the two agents exactly as in Notebook 03: EfficientNet-B4 (Agent A) and ViT-B/16 (Agent B) via `timm`, loading their trained checkpoints with `find_file`. The sentence encoder has been removed (no more text embeddings).

In [ ]:
import timm

AGENT_A_MODEL = "efficientnet_b4"
AGENT_B_MODEL = "vit_base_patch16_224.augreg_in21k_ft_in1k"


def load_agent(model_name, ckpt_substring):
    """Create a timm backbone with an 8-class head and load its checkpoint."""
    ckpt_path = find_file(ckpt_substring)
    if ckpt_path is not None and os.path.exists(ckpt_path):
        model = timm.create_model(model_name, pretrained=False, num_classes=NUM_CLASSES)
        state = torch.load(ckpt_path, map_location=DEVICE)
        if isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]
        res = model.load_state_dict(state, strict=False)
        print("Loaded %s from %s (missing=%d, unexpected=%d)" % (
            model_name, ckpt_path, len(res.missing_keys), len(res.unexpected_keys)))
    else:
        print("WARNING: no checkpoint for %s; using ImageNet-pretrained backbone "
              "with a random 8-class head (predictions NOT clinically meaningful)." % model_name)
        model = timm.create_model(model_name, pretrained=True, num_classes=NUM_CLASSES)
    model.eval().to(DEVICE)
    return model


agent_a = load_agent(AGENT_A_MODEL, "agent_a_best")
agent_b = load_agent(AGENT_B_MODEL, "agent_b_best")


## 4. Dataset, the SAME stratified train split, and the debate trigger

We build the ISIC-2019 dataset from the one-hot ground-truth CSV (label = argmax over the 8
class columns), then create the **same** stratified train/val split used everywhere
(`test_size=0.15, stratify=labels, random_state=42`). We then run both agents over the TRAIN
images and select those where the **debate trigger fires** (squared Jensen-Shannon divergence
> tau, or either agent's Shannon entropy > tau). Optionally we restrict to Notebook 03's
`hard_subset.csv` when it is attached.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

BATCH_SIZE = 32  # lower to 16 if you hit CUDA OOM
NUM_WORKERS = 2

CSV_PATH, IMAGE_DIR = discover_isic()

df = pd.read_csv(CSV_PATH)
# image-id column is named "image" in the ISIC-2019 ground truth.
id_col = "image" if "image" in df.columns else df.columns[0]
labels_all = df[ISIC_CLASSES].values.argmax(axis=1).astype(np.int64)
df = df.assign(_label=labels_all)
print("Total labelled rows:", len(df))
print("Class counts:", {ISIC_CLASSES[i]: int((labels_all == i).sum()) for i in range(NUM_CLASSES)})

# The SAME stratified split used across the project.
train_df, val_df = train_test_split(
    df, test_size=0.15, stratify=df["_label"].values, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print("Train rows:", len(train_df), "Val rows:", len(val_df))

# Optionally restrict the TRAIN pool to Notebook 03's hard subset, if attached.
hard_csv = find_file("hard_subset.csv")
if hard_csv is not None:
    try:
        hard_df = pd.read_csv(hard_csv)
        hcol = "image" if "image" in hard_df.columns else hard_df.columns[0]
        hard_ids = set(hard_df[hcol].astype(str).tolist())
        before = len(train_df)
        train_df = train_df[train_df[id_col].astype(str).isin(hard_ids)].reset_index(drop=True)
        print("Restricted TRAIN pool to hard_subset.csv: %d -> %d rows" % (before, len(train_df)))
    except Exception as exc:
        print("Could not apply hard_subset.csv (%s); using full train split." % exc)


# val-style transform (deterministic) for trigger scan + feature building.
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# train-style augmentation transform (documented for completeness; the consensus
# head trains on extracted features, so feature extraction uses eval_tf).
train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.1),
])


class ISICDataset(Dataset):
    """Reads the one-hot ISIC-2019 ground-truth CSV; label = argmax over 8 classes."""

    def __init__(self, frame, image_dir, transform):
        self.frame = frame.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image_id = str(row[id_col])
        path = os.path.join(self.image_dir, image_id + ".jpg")
        img = Image.open(path).convert("RGB")
        tensor = self.transform(img)
        label = int(row["_label"])
        return tensor, label, path


# --- Trigger (inlined from backend/ml/debate/trigger.py) -------------------
from scipy.spatial.distance import jensenshannon

TAU_JS = 0.25        # squared Jensen-Shannon divergence threshold
TAU_ENTROPY = 0.8   # Shannon entropy (bits) threshold
_EPS = 1e-12


def shannon_entropy(probs):
    p = np.clip(np.asarray(probs, dtype=np.float64), 0.0, None)
    return -float(np.sum(p * np.log2(p + _EPS)))


def trigger_fired(pa, pb, tau_js=TAU_JS, tau_entropy=TAU_ENTROPY):
    pa = np.asarray(pa, dtype=np.float64).ravel()
    pb = np.asarray(pb, dtype=np.float64).ravel()
    jsd = jensenshannon(pa, pb, base=2)
    jsd = float(jsd) ** 2
    if not np.isfinite(jsd):
        jsd = 0.0
    ea = shannon_entropy(pa)
    eb = shannon_entropy(pb)
    fired = (jsd > tau_js) or (max(ea, eb) > tau_entropy)
    return fired, jsd, ea, eb


@torch.no_grad()
def agent_probs(model, tensor):
    """Return the 8-class softmax probability vector (numpy) for a (1,3,224,224) tensor."""
    logits = model(tensor.to(DEVICE))
    return F.softmax(logits, dim=1)[0].detach().cpu().numpy().astype(np.float64)


# Scan the TRAIN pool and keep images where the trigger fires.
# Cap the number of fired images so a Kaggle session finishes comfortably; raise
# MAX_FIRED if you have time/credits to spare.
MAX_SCAN = 8000     # how many train rows to scan for triggers
MAX_FIRED = 1500     # cap on the number of debate samples to build features for

scan_df = train_df.iloc[:min(MAX_SCAN, len(train_df))].reset_index(drop=True)
scan_ds = ISICDataset(scan_df, IMAGE_DIR, eval_tf)

fired_records = []  # each: dict(image_id, path, label, pa, pb, jsd)
print("Scanning %d train images for trigger firing..." % len(scan_ds))
for i in tqdm(range(len(scan_ds))):
    tensor, label, path = scan_ds[i]
    tensor = tensor.unsqueeze(0)
    pa = agent_probs(agent_a, tensor)
    pb = agent_probs(agent_b, tensor)
    fired, jsd, ea, eb = trigger_fired(pa, pb)
    if fired:
        fired_records.append({
            "image_id": os.path.splitext(os.path.basename(path))[0],
            "path": path,
            "label": int(label),
            "pa": pa,
            "pb": pb,
            "jsd": float(jsd),
        })
    if len(fired_records) >= MAX_FIRED:
        break

print("Trigger fired on %d images (building consensus features for these)." % len(fired_records))
assert len(fired_records) > 0, "No images fired the trigger; lower TAU_JS / TAU_ENTROPY or raise MAX_SCAN."


## 5. Build the 23-d consensus features for every fired image

For each fired image we compute **Grad-CAM++** (Agent A) and **attention rollout** (Agent B) saliency maps, then assemble the 23-dim vector via `extract_consensus_features` (pA, pB, JS divergence, entropies, max prob delta, attention IoU/entropies). No LLM debate, no sentence embeddings. Per-feature diagnostics are printed to catch degenerate columns.

In [ ]:
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# --- Grad-CAM++ for Agent A (target layer resolution mirrors the backend) ---
def _resolve_target_layer(model):
    blocks = getattr(model, "blocks", None)
    if blocks is not None and len(blocks) > 0:
        last_block = blocks[-1]
        bn3 = getattr(last_block, "bn3", None)
        if isinstance(bn3, nn.Module):
            return bn3
        return last_block
    conv_head = getattr(model, "conv_head", None)
    if isinstance(conv_head, nn.Module):
        return conv_head
    fallback = None
    for module in model.modules():
        if isinstance(module, (nn.BatchNorm2d, nn.Conv2d)):
            fallback = module
    if fallback is None:
        raise RuntimeError("No Grad-CAM++ target layer found.")
    return fallback


def compute_gradcam_plusplus(model, tensor, target_class):
    target_layer = _resolve_target_layer(model)
    model.zero_grad()
    cam = GradCAMPlusPlus(model=model, target_layers=[target_layer])
    grayscale = cam(input_tensor=tensor, targets=[ClassifierOutputTarget(int(target_class))])
    model.zero_grad()
    heat = np.asarray(grayscale[0], dtype=np.float32)
    if heat.shape != (IMAGE_SIZE, IMAGE_SIZE):
        heat = cv2.resize(heat, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_LINEAR)
    return np.ascontiguousarray(np.clip(heat, 0.0, 1.0), dtype=np.float32)


# --- Attention rollout for Agent B (inlined from backend/ml/attention/rollout.py) ---
GRID_SIZE = 14
NUM_PATCH_TOKENS = GRID_SIZE * GRID_SIZE


def _centered_gaussian(size=IMAGE_SIZE, sigma_frac=0.25):
    coords = np.linspace(-1.0, 1.0, size, dtype=np.float32)
    gx, gy = np.meshgrid(coords, coords)
    sigma = max(sigma_frac, 1e-6)
    g = np.exp(-(gx ** 2 + gy ** 2) / (2.0 * sigma * sigma)).astype(np.float32)
    span = float(g.max() - g.min())
    if span < 1e-12:
        return np.zeros((size, size), dtype=np.float32)
    return ((g - g.min()) / span).astype(np.float32)


def compute_attention_rollout(model, tensor):
    handles = []
    captured = {}
    original_fused = {}
    blocks = getattr(model, "blocks", None)
    if blocks is None or len(blocks) == 0:
        return _centered_gaussian()

    def _make_hook(layer_idx):
        def _hook(module, inputs, output):
            try:
                x = inputs[0]
                batch, num_tokens, dim = x.shape
                num_heads = int(module.num_heads)
                head_dim = dim // num_heads
                scale = getattr(module, "scale", None)
                if scale is None:
                    scale = head_dim ** -0.5
                qkv = module.qkv(x).reshape(batch, num_tokens, 3, num_heads, head_dim)
                qkv = qkv.permute(2, 0, 3, 1, 4)
                q, k = qkv[0], qkv[1]
                attn = (q @ k.transpose(-2, -1)) * float(scale)
                attn = attn.softmax(dim=-1)
                captured[layer_idx] = attn.mean(dim=1).detach().to(torch.float32).cpu()
            except Exception:
                pass
        return _hook

    try:
        for idx, block in enumerate(blocks):
            attn_module = getattr(block, "attn", None)
            if attn_module is None or not hasattr(attn_module, "qkv"):
                continue
            original_fused[idx] = bool(getattr(attn_module, "fused_attn", False))
            if hasattr(attn_module, "fused_attn"):
                attn_module.fused_attn = False
            handles.append(attn_module.register_forward_hook(_make_hook(idx)))
        if not handles:
            return _centered_gaussian()
        with torch.no_grad():
            model(tensor)
        if not captured:
            return _centered_gaussian()
        layer_indices = sorted(captured.keys())
        num_tokens = captured[layer_indices[0]].shape[-1]
        identity = torch.eye(num_tokens, dtype=torch.float32)
        rollout = torch.eye(num_tokens, dtype=torch.float32)
        for idx in layer_indices:
            attn = captured[idx][0]
            aug = 0.5 * attn + 0.5 * identity
            aug = aug / aug.sum(dim=-1, keepdim=True).clamp_min(1e-12)
            rollout = aug @ rollout
        cls_attention = rollout[0, 1:]
        if cls_attention.shape[0] < NUM_PATCH_TOKENS:
            return _centered_gaussian()
        grid = cls_attention[:NUM_PATCH_TOKENS].reshape(GRID_SIZE, GRID_SIZE).numpy().astype(np.float32)
        heat = cv2.resize(grid, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_CUBIC).astype(np.float32)
        span = float(heat.max() - heat.min())
        if span < 1e-12:
            return _centered_gaussian()
        heat = (heat - heat.min()) / span
        return np.ascontiguousarray(np.clip(heat, 0.0, 1.0), dtype=np.float32)
    except Exception:
        return _centered_gaussian()
    finally:
        for handle in handles:
            handle.remove()
        for idx, was_fused in original_fused.items():
            attn_module = getattr(blocks[idx], "attn", None)
            if attn_module is not None and hasattr(attn_module, "fused_attn"):
                attn_module.fused_attn = was_fused

print("Attention helpers ready (Grad-CAM++ for Agent A, rollout for Agent B).")


In [ ]:
# --- Canonical 23-dim consensus feature extractor -------------------------
# IDENTICAL to ml_training/debate_text_utils.py, backend/ml/debate/features.py and
# 05_evaluation.ipynb. If you change this you MUST change it in all four places and
# retrain the consensus MLP, else train/eval/serve features silently disagree.
from scipy.stats import entropy as scipy_entropy
from scipy.spatial.distance import jensenshannon

CLASS_NAMES = ISIC_CLASSES

FEATURE_NAMES = (
    [f"pA_{c}" for c in CLASS_NAMES]
    + [f"pB_{c}" for c in CLASS_NAMES]
    + ["js_div", "entropy_a", "entropy_b", "max_prob_delta",
       "attn_iou", "attn_entropy_a", "attn_entropy_b"]
)


def extract_consensus_features(prob_a, prob_b, attn_map_a=None, attn_map_b=None):
    """Return the 23-d consensus feature vector for one image.

    Layout: [pA(8), pB(8), js_div, entropy_a, entropy_b, max_prob_delta,
             attn_iou, attn_entropy_a, attn_entropy_b]. The three attention
    features are 0.0 when either attention map is None.
    """
    prob_a = np.asarray(prob_a, dtype=np.float64).flatten()
    prob_b = np.asarray(prob_b, dtype=np.float64).flatten()

    prob_a = np.clip(prob_a, 1e-9, 1.0)
    prob_a /= prob_a.sum()
    prob_b = np.clip(prob_b, 1e-9, 1.0)
    prob_b /= prob_b.sum()

    js_div = float(jensenshannon(prob_a, prob_b) ** 2)
    entropy_a = float(scipy_entropy(prob_a, base=2))
    entropy_b = float(scipy_entropy(prob_b, base=2))
    max_prob_delta = float(np.max(np.abs(prob_a - prob_b)))

    if attn_map_a is not None and attn_map_b is not None:
        a = np.asarray(attn_map_a, dtype=np.float32)
        b = np.asarray(attn_map_b, dtype=np.float32)
        a = (a - a.min()) / (a.max() - a.min() + 1e-9)
        b = (b - b.min()) / (b.max() - b.min() + 1e-9)
        mask_a = (a >= 0.5).astype(np.float32)
        mask_b = (b >= 0.5).astype(np.float32)
        intersection = (mask_a * mask_b).sum()
        union = np.clip(mask_a + mask_b, 0, 1).sum()
        attn_iou = float(intersection / (union + 1e-9))
        a_flat = a.flatten() + 1e-9
        a_flat /= a_flat.sum()
        b_flat = b.flatten() + 1e-9
        b_flat /= b_flat.sum()
        attn_entropy_a = float(scipy_entropy(a_flat, base=2))
        attn_entropy_b = float(scipy_entropy(b_flat, base=2))
    else:
        attn_iou = 0.0
        attn_entropy_a = 0.0
        attn_entropy_b = 0.0

    feat = np.concatenate([
        prob_a,
        prob_b,
        [js_div, entropy_a, entropy_b, max_prob_delta,
         attn_iou, attn_entropy_a, attn_entropy_b],
    ]).astype(np.float32)

    assert feat.shape == (FEATURE_DIM,), \
        f"Feature dim mismatch: got {feat.shape}, expected ({FEATURE_DIM},)"
    assert not np.any(np.isnan(feat)), "NaN in feature vector"
    assert not np.any(np.isinf(feat)), "Inf in feature vector"
    return feat


print("extract_consensus_features ready (FEATURE_DIM=%d)." % FEATURE_DIM)


In [ ]:
# --- Build the 23-d feature matrix + labels over all fired images ----------
print("Building 23-dim consensus features for %d fired images..." % len(fired_records))

feat_eval_ds = ISICDataset(
    pd.DataFrame({id_col: [r["image_id"] for r in fired_records],
                  "_label": [r["label"] for r in fired_records]}),
    IMAGE_DIR, eval_tf)

X_list, y_list = [], []
attn_fail = 0
for i, rec in enumerate(tqdm(fired_records)):
    tensor, label, _path = feat_eval_ds[i]
    tensor = tensor.unsqueeze(0).to(DEVICE)
    pa = rec["pa"]; pb = rec["pb"]
    try:
        cam_input = tensor.clone().requires_grad_(True)
        heat_a = compute_gradcam_plusplus(agent_a, cam_input, int(np.argmax(pa)))
        heat_b = compute_attention_rollout(agent_b, tensor)
    except Exception as exc:
        attn_fail += 1
        print("  attention failed for %s (%s); attention features set to 0.0" % (rec["image_id"], exc))
        heat_a, heat_b = None, None
    feat = extract_consensus_features(pa, pb, heat_a, heat_b)
    X_list.append(feat)
    y_list.append(int(label))

X = np.stack(X_list).astype(np.float32)
y = np.array(y_list, dtype=np.int64)
if attn_fail:
    print("WARNING: attention maps failed for %d image(s); their 3 attention features are 0.0." % attn_fail)

# --- Verify the feature matrix (shape, NaN/Inf, degenerate columns) --------
print("Feature matrix:", X.shape, " labels:", y.shape)
assert X.shape[1] == 23, "Expected 23 features, got %d" % X.shape[1]
assert not np.any(np.isnan(X)), "NaN found in feature matrix"
assert not np.any(np.isinf(X)), "Inf found in feature matrix"

print("\nPer-feature mean / std (a constant column, std~0, would indicate a bug):")
for i, name in enumerate(FEATURE_NAMES):
    print("  %-16s mean=%.4f  std=%.4f" % (name, X[:, i].mean(), X[:, i].std()))

print("\nSamples per class in the new 23-dim consensus training set (audit 3.4):")
for ci, cname in enumerate(ISIC_CLASSES):
    print("  %-4s %d" % (cname, int((y == ci).sum())))

# --- Stratified split; fit StandardScaler on TRAIN only (no leakage) -------
from sklearn.model_selection import train_test_split
try:
    feat_train_idx, feat_val_idx = train_test_split(
        np.arange(len(y)), test_size=0.20, stratify=y, random_state=42)
except ValueError as exc:
    print("Stratified split failed (%s); falling back to a non-stratified split." % exc)
    feat_train_idx, feat_val_idx = train_test_split(
        np.arange(len(y)), test_size=0.20, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X[feat_train_idx])
X_val = scaler.transform(X[feat_val_idx])
y_train = y[feat_train_idx]
y_val = y[feat_val_idx]
print("\nTrain:", X_train.shape, " Val:", X_val.shape,
      "  (scaler fit on the training split only -- no data leakage)")

joblib.dump(scaler, "/kaggle/working/consensus_scaler.pkl")
with open("/kaggle/working/consensus_scaler.json", "w") as f:
    json.dump({"mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist()}, f)
print("Saved scaler -> consensus_scaler.pkl (+ consensus_scaler.json sidecar for the backend)")

# Tensors consumed by the training / temperature-scaling / ECE cells below.
Xtr = torch.tensor(X_train, dtype=torch.float32); ytr = torch.tensor(y_train, dtype=torch.long)
Xva = torch.tensor(X_val, dtype=torch.float32);   yva = torch.tensor(y_val, dtype=torch.long)


## 6. Define and train the consensus MLP (23 -> 128 -> 64 -> 8)

Architecture **must match** `backend/ml/consensus/classifier.py` (param names `mlp.*` + `temperature`):

```
Linear(23, 128) -> BatchNorm1d(128) -> ReLU -> Dropout(0.3)
Linear(128, 64) -> BatchNorm1d(64)  -> ReLU -> Dropout(0.3)
Linear(64, 8)
```

Training: Adam (lr=1e-3) on `CrossEntropyLoss` with **sqrt-inverse-frequency, clamped** class weights (the validated scheme; NOT raw inverse frequency) and label smoothing 0.1. `ReduceLROnPlateau` (patience 10) on **validation balanced accuracy**, early stopping (patience 20), max 200 epochs, batch size 64. Temperature is held at 1.0 during MLP training and calibrated afterwards.

In [ ]:
from torch.utils.data import TensorDataset
from sklearn.metrics import balanced_accuracy_score
from collections import Counter


class ConsensusClassifier(nn.Module):
    """23-dim consensus MLP. Parameter names (mlp.*, temperature) match
    backend/ml/consensus/classifier.py so the checkpoint loads in the app."""

    def __init__(self, input_dim=FEATURE_DIM, num_classes=NUM_CLASSES, dropout=0.3):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes),
        )
        self.temperature = nn.Parameter(torch.ones(1))

    def logits(self, x):
        return self.mlp(x)

    def forward(self, x):
        return self.logits(x) / torch.clamp(self.temperature, min=1e-2)


# --- Feature-quality guard before training --------------------------------
if not (np.isfinite(X_train).all() and np.isfinite(X_val).all()):
    raise ValueError("Scaled feature matrix contains NaN/Inf!")
assert X_train.shape[1] == 23, "Expected 23 features, got %d" % X_train.shape[1]

train_ds = TensorDataset(Xtr, ytr)
val_ds = TensorDataset(Xva, yva)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

model = ConsensusClassifier().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print("ConsensusClassifier: %d parameters | Input: 23 | Output: 8" % total_params)

# sqrt-inverse-frequency, clamped class weights (validated scheme; NOT raw inverse).
counts = Counter(ytr.numpy().tolist())
total_train = len(ytr)
raw_w = [total_train / (NUM_CLASSES * max(1, counts.get(i, 0))) for i in range(NUM_CLASSES)]
sqrt_w = np.sqrt(raw_w)
clamp_w = np.clip(sqrt_w, a_min=None, a_max=5.0 * sqrt_w.min())
class_weights_tensor = torch.tensor(clamp_w, dtype=torch.float32).to(DEVICE)
print("Consensus class weights:",
      {ISIC_CLASSES[i]: round(float(clamp_w[i]), 3) for i in range(NUM_CLASSES)})
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1)

optimizer = torch.optim.Adam(model.mlp.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=10, factor=0.5)

MAX_EPOCHS = 200
PATIENCE = 20
best_bal_acc = -1.0
best_epoch = -1
best_state = None
epochs_no_improve = 0
history = {"train_loss": [], "val_loss": [], "val_bal_acc": []}

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    running = 0.0
    for xb, yb in train_loader:
        if xb.shape[0] < 2:   # BatchNorm needs >1 sample.
            continue
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model.logits(xb), yb)
        loss.backward()
        optimizer.step()
        running += loss.item() * xb.shape[0]
    train_loss = running / max(1, len(train_ds))

    model.eval()
    preds, vloss, tot = [], 0.0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            out = model.logits(xb.to(DEVICE))
            vloss += criterion(out, yb.to(DEVICE)).item() * xb.shape[0]
            tot += xb.shape[0]
            preds.extend(out.argmax(1).cpu().numpy())
    val_loss = vloss / max(1, tot)
    val_bal_acc = balanced_accuracy_score(y_val, preds)
    scheduler.step(val_bal_acc)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_bal_acc"].append(val_bal_acc)

    if val_bal_acc > best_bal_acc:
        best_bal_acc = val_bal_acc
        best_epoch = epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if epoch % 10 == 0 or epochs_no_improve == 0:
        print("Epoch %3d/%d  train_loss=%.4f  val_loss=%.4f  val_bal_acc=%.4f  best=%.4f  no_improve=%d"
              % (epoch, MAX_EPOCHS, train_loss, val_loss, val_bal_acc, best_bal_acc, epochs_no_improve))

    if epochs_no_improve >= PATIENCE:
        print("\nEarly stopping at epoch %d. Best val balanced accuracy: %.4f (epoch %d)."
              % (epoch, best_bal_acc, best_epoch))
        break

if best_state is not None:
    model.load_state_dict(best_state)
print("\nTraining complete. Best validation balanced accuracy: %.4f (epoch %d)"
      % (best_bal_acc, best_epoch))


### Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Consensus MLP loss"); axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend()
axes[1].plot(history["val_bal_acc"], color="green")
axes[1].set_title("Consensus MLP val balanced accuracy"); axes[1].set_xlabel("epoch"); axes[1].set_ylabel("balanced accuracy")
plt.tight_layout()
os.makedirs("/kaggle/working/figures", exist_ok=True)
plt.savefig("/kaggle/working/figures/consensus_training_curves.png", dpi=120, bbox_inches="tight")
plt.show()


## 7. Temperature scaling (post-hoc calibration)

We freeze the MLP and optimise the single scalar `temperature` on the **validation NLL** using
**LBFGS**. This rescales the logits so the predicted confidences are better calibrated without
changing the argmax decision.

In [ ]:
# Freeze the MLP; only the temperature parameter is optimised.
for p in model.mlp.parameters():
    p.requires_grad_(False)
model.temperature.requires_grad_(True)
model.eval()  # keep BatchNorm in eval mode using running stats

with torch.no_grad():
    val_logits = model.logits(Xva.to(DEVICE)).detach()
    val_targets = yva.to(DEVICE)

nll = nn.CrossEntropyLoss()
optimizer_t = torch.optim.LBFGS([model.temperature], lr=0.01, max_iter=100)


def _closure():
    optimizer_t.zero_grad()
    t = torch.clamp(model.temperature, min=1e-2)
    loss = nll(val_logits / t, val_targets)
    loss.backward()
    return loss


before = nll(val_logits, val_targets).item()
optimizer_t.step(_closure)
learned_temperature = float(torch.clamp(model.temperature, min=1e-2).detach().cpu().item())
with torch.no_grad():
    after = nll(val_logits / learned_temperature, val_targets).item()

print("Validation NLL before scaling: %.4f" % before)
print("Validation NLL after  scaling: %.4f" % after)
print("Learned temperature: %.4f" % learned_temperature)


## 8. Expected Calibration Error (ECE) and validation report

We compute ECE with `netcal.metrics.ECE` on the temperature-scaled validation probabilities,
before vs. after calibration, and show the confusion matrix and per-class report.

In [ ]:
from netcal.metrics import ECE
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

with torch.no_grad():
    probs_before = F.softmax(val_logits, dim=1).cpu().numpy()
    probs_after = F.softmax(val_logits / learned_temperature, dim=1).cpu().numpy()
targets_np = yva.numpy()

ece_metric = ECE(bins=15)
ece_before = float(ece_metric.measure(probs_before, targets_np))
ece_after = float(ece_metric.measure(probs_after, targets_np))
print("ECE before temperature scaling: %.4f" % ece_before)
print("ECE after  temperature scaling: %.4f" % ece_after)

preds_after = probs_after.argmax(axis=1)
val_accuracy = accuracy_score(targets_np, preds_after)
print("Calibrated val accuracy: %.4f" % val_accuracy)

present = sorted(set(targets_np.tolist()) | set(preds_after.tolist()))
print("\nClassification report (validation):")
print(classification_report(
    targets_np, preds_after,
    labels=present,
    target_names=[ISIC_CLASSES[i] for i in present],
    zero_division=0,
))

cm = confusion_matrix(targets_np, preds_after, labels=list(range(NUM_CLASSES)))
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=ISIC_CLASSES, yticklabels=ISIC_CLASSES)
plt.title("Consensus MLP confusion matrix (val)")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout()
plt.savefig("/kaggle/working/figures/consensus_confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()

# Reliability diagram (calibrated).
plt.figure(figsize=(6, 5))
conf = probs_after.max(axis=1)
correct = (preds_after == targets_np).astype(float)
bins = np.linspace(0, 1, 11)
idx = np.digitize(conf, bins) - 1
acc_per_bin, conf_per_bin, centers = [], [], []
for b in range(10):
    m = idx == b
    if m.sum() > 0:
        acc_per_bin.append(correct[m].mean())
        conf_per_bin.append(conf[m].mean())
        centers.append((bins[b] + bins[b + 1]) / 2)
plt.plot([0, 1], [0, 1], "k--", label="perfect")
plt.plot(conf_per_bin, acc_per_bin, "o-", label="calibrated")
plt.xlabel("Confidence"); plt.ylabel("Accuracy"); plt.title("Reliability diagram (calibrated)")
plt.legend(); plt.tight_layout()
plt.savefig("/kaggle/working/figures/consensus_reliability.png", dpi=120, bbox_inches="tight")
plt.show()


## 9. Save the checkpoint (state_dict incl. temperature) and the scalar temperature

We save the full `state_dict` (the `mlp.*` weights **and** the `temperature` parameter) to
`/kaggle/working/consensus_best.pth`, plus the scalar temperature to
`/kaggle/working/consensus_temperature.txt`. The checkpoint loads directly into the app's
`backend/ml/consensus/classifier.py` `ConsensusClassifier` because the architecture and
parameter names match exactly.

In [ ]:
# Make sure the learned temperature is the value saved.
with torch.no_grad():
    model.temperature.copy_(torch.tensor([learned_temperature], dtype=torch.float32, device=model.temperature.device))

CKPT_PATH = "/kaggle/working/consensus_best.pth"
TEMP_PATH = "/kaggle/working/consensus_temperature.txt"

# Save as an envelope so the app can pick up the ECE alongside the weights; the
# app's ConsensusClassifier handles both {"state_dict": ..., "ece": ...} and a
# bare state_dict via strict=False.
state_dict = model.state_dict()  # includes mlp.* and temperature
torch.save({"state_dict": state_dict, "ece": ece_after}, CKPT_PATH)

with open(TEMP_PATH, "w") as f:
    f.write("%.6f\n" % learned_temperature)

print("Saved checkpoint:", CKPT_PATH)
assert os.path.exists("/kaggle/working/consensus_scaler.pkl"), "consensus_scaler.pkl missing!"
print("Scaler present: consensus_scaler.pkl (+ .json sidecar) -- REQUIRED by NB05 and the backend.")
print("Saved temperature:", TEMP_PATH, "->", learned_temperature)
print("state_dict keys:", list(state_dict.keys()))

# Sanity check: reload into a fresh model exactly as the app does (strict=False).
_check = ConsensusClassifier()
_loaded = torch.load(CKPT_PATH, map_location="cpu")
_sd = _loaded["state_dict"] if isinstance(_loaded, dict) and "state_dict" in _loaded else _loaded
_res = _check.load_state_dict(_sd, strict=False)
print("Reload check -> missing:", _res.missing_keys, " unexpected:", _res.unexpected_keys)
assert len(_res.missing_keys) == 0 and len(_res.unexpected_keys) == 0, "Checkpoint will not load cleanly in the app!"
print("Checkpoint loads cleanly into a fresh ConsensusClassifier (matches the app contract).")


## 10. Outputs and how to use them

All artifacts are written to **`/kaggle/working/`**:

| File | Purpose |
|------|---------|
| `consensus_best.pth` | Trained 23-dim consensus MLP `state_dict` (incl. `temperature`) + `ece`. |
| `consensus_scaler.pkl` | StandardScaler fit on the consensus training split. **Required** by NB05 and the backend. |
| `consensus_scaler.json` | `{"mean", "scale"}` sidecar so the backend needs only numpy. |
| `consensus_temperature.txt` | The scalar calibration temperature. |
| `figures/*.png` | Training curves, confusion matrix, reliability diagram. |

### Hand off to NB05 (evaluation)
Save a version so `/kaggle/working` is published, then in NB05 *Add Data -> Notebook Output* and attach it. `find_file('consensus_best')` and `find_file('consensus_scaler')` locate the checkpoint and scaler. The scaler **must** be attached or NB05/the backend will apply an identity transform and the predictions will be miscalibrated.

> The Groq response cache and the `.npy` feature caches are gone: feature building is now cheap enough (no LLM calls) to run every time.